In [1]:
%pip install azure-eventhub

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 7, Finished, Available, Finished, True)


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: /nfs4/pyenv-76b2fe62-99a5-41d3-bc39-5ef41b5f9342/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.



In [2]:
#importing libraries
import pandas as pd
import json
from datetime import datetime, timezone
import time
import random
import random

from azure.eventhub import EventHubProducerClient, EventData


StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 9, Finished, Available, Finished, False)

In [3]:
# importing functions
from pyspark.sql.functions import (
    col,
    current_timestamp,
    expr,
    lit,
    struct,
    to_json
)

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 10, Finished, Available, Finished, False)

In [5]:
#read the data and show top 5
df = pd.read_csv("abfss://82bb17bd-3109-48ac-b62a-96133084d6ae@onelake.dfs.fabric.microsoft.com/e46cb4a0-4ee8-4483-b0bb-9a8c3c4d2d23/Files/finance_banking_transactions_2022_2025.csv")

display(df.head(5))

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 76c2df28-f73e-466f-b60c-cf5bd3fcb16c)

In [6]:
# connect to eventstream custom endpoint
CONNECTION_STRING = (
    
    )

# Create an Event Hubs producer using the Fabric-generated Custom Endpoint connection string.

producer = EventHubProducerClient.from_connection_string(
    conn_str=CONNECTION_STRING
)


StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 16, Finished, Available, Finished, False)

In [7]:
def create_event(row):


    # Convert the selected Pandas DataFrame row

    event = row.to_dict()


    # Add the current UTC timestamp.


    event["event_timestamp"] = (
        datetime.now(timezone.utc).isoformat()
    )

    # Convert Pandas values into values that
    # can safely be serialized into JSON.
 

    for column, value in event.items():

        # Convert missing Pandas values into JSON null.
        if pd.isna(value):

            event[column] = None


        # Convert NumPy scalar types to standard
        # Python values.
        elif hasattr(value, "item"):

            event[column] = value.item()


    # Return the generated event.
    return event

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 17, Finished, Available, Finished, False)

In [8]:
#------------------------
#TEST 1 - Streaming data print output in notebook
#--------------------------

# Randomly select one transaction from the DataFrame.

preview_row = (
    df.sample(n=1)
      .iloc[0]
)


# Convert it into a live event.

preview_event = create_event(
    preview_row
)


# Display the generated transaction as JSON.

print(
    json.dumps(
        preview_event,
        indent=2,
        default=str
    )
)

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 18, Finished, Available, Finished, False)

{
  "transaction_id": "TXN-20230002597",
  "account_id": "ACC-253175704",
  "customer_id": "CUST-956196",
  "customer_name": "Lukas Meyer",
  "account_type": "Mortgage",
  "account_status": "Active",
  "transaction_type": "Refund",
  "transaction_date": "2022-06-29",
  "amount": -33686.71,
  "currency": "CHF",
  "balance_after": 184465.47,
  "transaction_channel": "ATM",
  "branch_city": "Nuremberg",
  "merchant_name": "Shell",
  "payment_method": "SEPA",
  "is_international": true,
  "fee_amount": 13.84,
  "risk_score": 37.93,
  "fraud_flag": false,
  "aml_check_status": "Passed",
  "kyc_status": "Verified",
  "credit_score": null,
  "customer_segment": "Private Banking",
  "ip_address": "142.251.166.59",
  "created_at": "2023-10-08 21:32:00",
  "event_timestamp": "2026-11-05T08:53:26.509822+00:00"
}


In [9]:
def send_to_eventstream(event):


    # Convert the Python transaction dictionary into JSON.

    event_json = json.dumps(
        event,
        default=str
    )

    # Create a new Event Hubs batch.

    batch = producer.create_batch()

    # Add the JSON transaction to the batch.

    batch.add(
        EventData(event_json)
    )


    # Send the batch to the Fabric Eventstream
    # Custom Endpoint.

    producer.send_batch(batch)

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 19, Finished, Available, Finished, False)

Testing data streaming in eventstreams

In [10]:
# Select one random banking transaction.

test_row = (
    df.sample(n=1)
      .iloc[0]
)


# Convert the historical row into a live event.

test_event = create_event(
    test_row
)


# Send the transaction to Fabric Eventstream.

send_to_eventstream(
    test_event
)


# Show what was transmitted.

print(
    "Test event sent successfully:",
    test_event.get("transaction_id"),
    "|",
    test_event.get("transaction_channel"),
    "|",
    test_event.get("amount"),
    test_event.get("currency"),
    "|",
    test_event.get("event_timestamp")
)

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 20, Finished, Available, Finished, False)

Test event sent successfully: TXN-20230008705 | Branch | -11355.39 CHF | 2026-11-05T08:53:52.166272+00:00


In [11]:
# continous streaming of data

stream_transactions(
    dataframe=df,
    number_of_events=100,
    minimum_delay=0.5,
    maximum_delay=1.5
)

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 21, Finished, Available, Finished, False)

NameError: name 'stream_transactions' is not defined

In [12]:
#Test the issue/error

print("df exists:", "df" in globals())
print("producer exists:", "producer" in globals())
print("create_event exists:", "create_event" in globals())
print("send_to_eventstream exists:", "send_to_eventstream" in globals())
print("stream_transactions exists:", "stream_transactions" in globals())

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 22, Finished, Available, Finished, False)

df exists: True
producer exists: True
create_event exists: True
send_to_eventstream exists: True
stream_transactions exists: False


In [13]:
# Stream every row in the DataFrame sequentially

def stream_transactions(df, delay_seconds=1):

    total_rows = len(df)

    print(f"Starting stream of {total_rows} transactions...")

    # Loop through every row in the DataFrame
    for i, (_, row) in enumerate(df.iterrows(), start=1):

        # Turn the current DataFrame row into a live event
        event = create_event(row)

        # Send the event to Fabric Eventstream
        send_to_eventstream(event)

        # Display progress
        print(
            f"Sent {i}/{total_rows}:",
            event.get("transaction_id")
        )

        # Wait before sending the next transaction
        time.sleep(delay_seconds)

    print("All transactions streamed successfully.")

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 23, Finished, Available, Finished, False)

In [14]:
stream_transactions(
    df,
    delay_seconds=1
)

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, 24, Submitted, Running, Running, True)

Starting stream of 52624 transactions...
Sent 1/52624: TXN-20230000000
Sent 2/52624: TXN-20230000001
Sent 3/52624: TXN-20230000002
Sent 4/52624: TXN-20230000003
Sent 5/52624: TXN-20230000004
Sent 6/52624: TXN-20230000005
Sent 7/52624: TXN-20230000006
Sent 8/52624: TXN-20230000007
Sent 9/52624: TXN-20230000008
Sent 10/52624: TXN-20230000009
Sent 11/52624: TXN-20230000010
Sent 12/52624: TXN-20230000011
Sent 13/52624: TXN-20230000012
Sent 14/52624: TXN-20230000013
Sent 15/52624: TXN-20230000014
Sent 16/52624: TXN-20230000015
Sent 17/52624: TXN-20230000016
Sent 18/52624: TXN-20230000017
Sent 19/52624: TXN-20230000018
Sent 20/52624: TXN-20230000019
Sent 21/52624: TXN-20230000020
Sent 22/52624: TXN-20230000021
Sent 23/52624: TXN-20230000022
Sent 24/52624: TXN-20230000023
Sent 25/52624: TXN-20230000024
Sent 26/52624: TXN-20230000025
Sent 27/52624: TXN-20230000026
Sent 28/52624: TXN-20230000027
Sent 29/52624: TXN-20230000028
Sent 30/52624: TXN-20230000029
Sent 31/52624: TXN-20230000030
Sent 32

Cancelled: Session has been cancelled.

In [9]:
# STREAMING 100 rows

def stream_transactions(df, number_of_events=1000, delay_seconds=1):

    for i in range(number_of_events):

        # Select a random banking transaction
        row = df.sample(n=1).iloc[0]

        # Turn it into a live event
        event = create_event(row)

        # Send it to Fabric Eventstream
        send_to_eventstream(event)

        print(
            f"Sent {i + 1}/{number_of_events}:",
            event.get("transaction_id")
        )


StatementMeta(, d78f5465-918d-4e8a-a07d-4a4abe81a2b0, 18, Finished, Available, Finished, False)

In [12]:
stream_transactions(
    df,
    number_of_events=1000,
    delay_seconds=1
)

StatementMeta(, d78f5465-918d-4e8a-a07d-4a4abe81a2b0, 23, Finished, Available, Finished, False)

Sent 1/1000: TXN-20230041721
Sent 2/1000: TXN-20230023194
Sent 3/1000: TXN-20230044503
Sent 4/1000: TXN-20230018192
Sent 5/1000: TXN-20230026708
Sent 6/1000: TXN-20230030295
Sent 7/1000: TXN-20230022189
Sent 8/1000: TXN-20230046214
Sent 9/1000: TXN-20230002288
Sent 10/1000: TXN-20230019790
Sent 11/1000: TXN-20230003292
Sent 12/1000: TXN-20230029569
Sent 13/1000: TXN-20230025651
Sent 14/1000: TXN-20230029704
Sent 15/1000: TXN-20230004485
Sent 16/1000: TXN-20230014100
Sent 17/1000: TXN-20230005594
Sent 18/1000: TXN-20230028741
Sent 19/1000: TXN-20230020739
Sent 20/1000: TXN-20230000155
Sent 21/1000: TXN-20230040242
Sent 22/1000: TXN-20230022675
Sent 23/1000: TXN-20230040286
Sent 24/1000: TXN-20230031186
Sent 25/1000: TXN-20230016586
Sent 26/1000: TXN-20230002034
Sent 27/1000: TXN-20230029599
Sent 28/1000: TXN-20230043712
Sent 29/1000: TXN-20230005970
Sent 30/1000: TXN-20230019167
Sent 31/1000: TXN-20230045399
Sent 32/1000: TXN-20230021949
Sent 33/1000: TXN-20230028797
Sent 34/1000: TXN-2

In [ ]:
#STREAM ALL DATA CONTINEOUSLY
def stream_all_transactions_continuously(
    dataframe,
    delay_seconds=1
):
    cycle = 1

    while True:

        print(f"Starting cycle {cycle}...")

        for index, row in dataframe.iterrows():

            # Create a new live event
            event = create_event(row)

            # Send to Fabric Eventstream
            send_to_eventstream(event)

            print(
                f"Cycle {cycle} | "
                f"Sent {index + 1}/{len(dataframe)} |",
                event.get("transaction_id")
            )

            # Wait before sending next transaction
            time.sleep(delay_seconds)

        cycle += 1

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# Close the Event Hubs producer connection cleanly.

producer.close()


print(
    "Eventstream producer closed successfully."
)

StatementMeta(, 46f06370-a5af-4089-b2af-a0b4215a693c, -1, Cancelled, , Cancelled, True)

In [ ]:
def stream_all_transactions(
    dataframe,
    delay_seconds=1
):
    # Total number of rows in the DataFrame
    total_rows = len(dataframe)

    print(f"Starting stream of all {total_rows} transactions...")

    # Loop through every row in the DataFrame
    for index, row in dataframe.iterrows():

        # Convert the historical transaction into a live event
        # and add the current event_timestamp
        event = create_event(row)

        # Send the transaction to Fabric Eventstream
        send_to_eventstream(event)

        # Show streaming progress
        print(
            f"Sent {index + 1}/{total_rows}:",
            event.get("transaction_id"),
            "|",
            event.get("transaction_channel"),
            "|",
            event.get("amount"),
            event.get("currency")
        )

        # Wait before sending the next transaction
        time.sleep(delay_seconds)

    print("All transactions streamed successfully.")

StatementMeta(, , -1, Waiting, , Waiting, True)